In [19]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator
import os

In [20]:
load_dotenv(find_dotenv())

True

In [21]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [22]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    overall_feedback: str
    avg_score: float

In [23]:
class Evaluation(BaseModel):
    feedback: str = Field(
        description="Detailed feedback for improvement"
    )
    score: int = Field(
        description="Score from 1 to 10"
    )

structured_model = model.with_structured_output(Evaluation)

In [24]:
def evaluate_language(state: UPSCState):

    prompt = f"""
Evaluate the language quality of this UPSC essay.

Check:
- grammar
- vocabulary
- sentence structure
- formal writing style

Essay:

{state['essay']}

Give feedback and a score out of 10.
"""


    output = structured_model.invoke(prompt)


    return {
        "language_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [25]:
def evaluate_analysis(state: UPSCState):

    prompt = f"""
Evaluate the depth of analysis in this UPSC essay.

Check:
- quality of arguments
- examples
- multidimensional thinking
- factual depth
- critical analysis

Essay:

{state['essay']}

Give feedback and a score out of 10.
"""


    output = structured_model.invoke(prompt)


    return {
        "analysis_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [26]:
def evaluate_clarity(state: UPSCState):

    prompt = f"""
Evaluate the clarity of thought in this UPSC essay.

Check:
- logical flow
- organization of ideas
- coherence
- connection between arguments

Essay:

{state['essay']}

Give feedback and a score out of 10.
"""


    output = structured_model.invoke(prompt)


    return {
        "clarity_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [27]:
def final_evaluation(state: UPSCState):


    avg = sum(state["individual_scores"]) / len(
        state["individual_scores"]
    )


    prompt = f"""

You are a UPSC essay evaluator.

Summarize the following feedback:

Language:
{state['language_feedback']}


Analysis:
{state['analysis_feedback']}


Clarity:
{state['clarity_feedback']}


Provide:
- overall strengths
- weaknesses
- improvement suggestions

"""


    response = model.invoke(prompt)


    return {
        "overall_feedback": response.content,
        "avg_score": avg
    }

In [28]:
graph = StateGraph(UPSCState)

graph.add_node(
    "evaluate_language",
    evaluate_language
)
graph.add_node(
    "evaluate_analysis",
    evaluate_analysis
)
graph.add_node(
    "evaluate_clarity",
    evaluate_clarity
)
graph.add_node(
    "final_evaluation",
    final_evaluation
)

graph.add_edge(
    START,
    "evaluate_language"
)

graph.add_edge(
    START,
    "evaluate_analysis"
)

graph.add_edge(
    START,
    "evaluate_clarity"
)


graph.add_edge(
    "evaluate_language",
    "final_evaluation"
)

graph.add_edge(
    "evaluate_analysis",
    "final_evaluation"
)

graph.add_edge(
    "evaluate_clarity",
    "final_evaluation"
)


graph.add_edge(
    "final_evaluation",
    END
)



workflow = graph.compile()

In [29]:
initial_state = {

    "essay": """
Artificial Intelligence is transforming India.
It has impacted healthcare, agriculture and education.
However, challenges like unemployment and ethics remain.
The government must ensure responsible AI adoption.
""",

    "individual_scores": []
}



final_state = workflow.invoke(initial_state)


print("Scores:", final_state["individual_scores"])

print(
    "Average Score:",
    final_state["avg_score"]
)

print(
    "Final Feedback:",
    final_state["overall_feedback"]
)

Scores: [3, 3, 5]
Average Score: 3.6666666666666665
Final Feedback: **Overall Strengths:**
- The essay presents a clear and concise argument about the impact of Artificial Intelligence (AI) on India, addressing relevant topics.
- The grammar is generally correct, showcasing basic proficiency in writing.

**Weaknesses:**
- The language used is basic and lacks sophistication, which detracts from the essay's depth.
- The arguments presented are superficial, lacking thorough exploration and specific examples.
- The discussion of challenges is limited and lacks critical analysis or potential solutions.
- The essay does not incorporate a multidimensional perspective, ignoring economic, social, and international collaboration aspects.
- The organization of ideas is simplistic, leading to a lack of logical flow and coherent transitions.

**Improvement Suggestions:**
- Enhance language complexity by employing more advanced vocabulary and varied sentence structures.
- Elaborate on the arguments 